- Date : 10-08-2026

### Step 1) Load the Data

In [13]:
import os
from langchain_community.document_loaders import TextLoader

data_path = "D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data"

text = []

for i in os.listdir(data_path):
    print(i)

    loader = TextLoader(os.path.join(data_path, i))

    text.extend(loader.load())

001ssb.txt
002ssb.txt
003ssb.txt
004ssb.txt
005ssb.txt


In [14]:
print("Document Length: ", len(text))
print("Document type: ", type(text[0]))

Document Length:  5
Document type:  <class 'langchain_core.documents.base.Document'>


### Step 2) Split the Data into Chunks

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

In [16]:
chunks = splitter.split_documents(text)

In [17]:
print("Chunks Length:", len(chunks))
print("Chunks type:", type(chunks))

Chunks Length: 23247
Chunks type: <class 'list'>


- Know that all chunks are loaded, and we didnt missed any chunk. Now we will check the source of each chunk and how many chunks are there for each source.


In [18]:
from collections import Counter

counts = Counter(
    chunk.metadata['source']
    for chunk in chunks
)

for source, count in counts.items():
    print(source, "→", count, "chunks")

D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\001ssb.txt → 3836 chunks
D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\002ssb.txt → 4242 chunks
D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\003ssb.txt → 5519 chunks
D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\004ssb.txt → 4041 chunks
D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\005ssb.txt → 5609 chunks


### Step 3) Create Embeddings for Each Chunk

In [19]:
# Define your embedding model

from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small"
)

In [20]:
# Define Your Vector Store

from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="my_documents",
    persist_directory="./my_chroma_db"
)

### Step 4) Search for Similar Chunks

In [24]:
results = vectorstore.similarity_search(
    "Who was daenerys targaryen",
    k = 3
)

for doc in results:
    print(doc.page_content)
    print(doc.metadata)
    print("----------------")

THE QUEEN ACROSS THE WATER 
DAENERYS TARGARYEN, called Daenerys Stormborn, the Unburnt, Mother of Dragons, Khaleesi 
of the Dothraki, and First of Her Name, sole surviving child of King Aerys II Targaryen by his sister/wife, 
Queen Rhaella, a widow at fourteen years, 
-her new-hatched dragons, DROGON, VISERION, RHAEGAL, 
-her brothers: 
-{RHAEGAR}, Prince of Dragonstone and heir to the Iron Throne, slain by King Robert on 
the Trident,
{'source': 'D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\\002ssb.txt'}
----------------
they made him king, he locked her up in a tower. His other sisters too. There was three.” 
“Daenela,” the proprietor said loudly. “That was her name. The Mad King’s daughter, I mean, 
not Baelor’s bloody wife.” 
“Daenerys,” Davos said. “She was named for the Daenerys who wed the Prince of Dorne during 
the reign of Daeron the Second. I don’t know what became of her.” 
“I do,” said the man who’d started all the talk of dragons, a Braa

In [26]:
results = vectorstore.similarity_search_with_score(
    "Who was Daenerys Targaryen?",
    k=3
)

for doc, score in results:
    print("Score:", score)
    print("Text:", doc.page_content)
    print("Metadata:", doc.metadata)
    print("----------------")

Score: 0.6501744985580444
Text: THE QUEEN ACROSS THE WATER 
DAENERYS TARGARYEN, called Daenerys Stormborn, the Unburnt, Mother of Dragons, Khaleesi 
of the Dothraki, and First of Her Name, sole surviving child of King Aerys II Targaryen by his sister/wife, 
Queen Rhaella, a widow at fourteen years, 
-her new-hatched dragons, DROGON, VISERION, RHAEGAL, 
-her brothers: 
-{RHAEGAR}, Prince of Dragonstone and heir to the Iron Throne, slain by King Robert on 
the Trident,
Metadata: {'source': 'D:/Sculptsoft/AI-ML_Sculptsoft/27-07-2026-Natural_Language_Processing/text_data\\002ssb.txt'}
----------------
Score: 0.7380026578903198
Text: THE QUEEN ACROSS THE WATER 
DAENERYS TARGARYEN, the First of Her Name, Queen of Meereen, Queen of the Andals and the 
Rhoynar and the First Men, Lord of the Seven Kingdoms, Protector of the Realm, Khaleesi of the Great 
Grass Sea, called DAENERYS STORMBORN, the UNBURNT, MOTHER OF DRAGONS, 
—her dragons, DROGON, VISERION, RHAEGAL, 
—her brother, {RHAEGAR}, Princ